# 15分钟城市时空可达性分析
## 南山区公共服务设施数据处理与时空贫困分析

**未来交通实验室 | 2026-05-20**

---

### 目录
1. [环境配置与依赖安装](#1-环境配置)
2. [数据加载与探索](#2-数据加载)
3. [POI数据清洗](#3-POI清洗)
4. [小区AOI数据处理](#4-小区AOI处理)
5. [设施类型标准化](#5-类型标准化)
6. [营业时间处理](#6-营业时间处理)
7. [空间分析](#7-空间分析)
8. [时间可达性计算](#8-可达性计算)
9. [时空贫困指数分析](#9-贫困指数分析)
10. [可视化与输出](#10-可视化输出)

---

<a id='1-环境配置'></a>
## 1. 环境配置与依赖安装

In [ ]:
# 安装必要依赖
!pip install pandas geopandas shapely pyproj matplotlib seaborn folium contextily osmnx scikit-learn -q
!pip install openpyxl xlrd -q  # Excel支持

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import math
import logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union
from pyproj import Transformer, CRS
from scipy.spatial import cKDTree
from scipy import stats

plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
    'SimSun', 'AR PL UMing CN', 'WenQuanYi Micro Hei', 'Arial Unicode MS', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

# 设置路径
BASE_DIR = Path(r'E:\xicha gis 智能定位\15分钟城市时间贫困研究')
OUTPUT_DIR = BASE_DIR / 'output'
DATA_DIR = BASE_DIR / 'osm_data'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"工作目录: {BASE_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

---

<a id='2-数据加载'></a>
## 2. 数据加载与探索

### 2.1 配置参数

In [ ]:
class StudyConfig:
    """研究配置类"""
    
    # 研究区域（南山区）
    STUDY_AREA = "南山区, 深圳市"
    BBOX = {
        'north': 22.54,
        'south': 22.48,
        'east': 113.98,
        'west': 113.85
    }
    
    # 步行参数
    WALK_SPEED_KMH = 5.0  # 步行速度 km/h
    ACCESSIBILITY_THRESHOLD = 15  # 15分钟可达阈值
    AVAILABLE_TIME = 120  # 单次可支配时间（分钟）
    
    # 时段定义
    PERIODS = {
        'day': {'name': '白天', 'start': 8, 'end': 18, 'description': '08:00-18:00'},
        'evening': {'name': '夜间', 'start': 18, 'end': 22, 'description': '18:00-22:00'},
        'night': {'name': '深夜', 'start': 22, 'end': 8, 'description': '22:00-次日08:00'}
    }
    
    # 曲折系数（不同社区类型）
    DETOUR_FACTORS = {
        'urban_village': 1.5,   # 城中村
        'old_community': 1.3,   # 老旧小区
        'normal_community': 1.1, # 普通商品房
        'high_end': 1.0         # 高端社区
    }
    
    # 服务耗时（分钟）
    SERVICE_DURATION = {
        'convenience_store': 5,
        'pharmacy': 10,
        'supermarket': 15,
        'hospital': 30,
        'clinic': 20,
        'bank': 45,
        'atm': 2,
        'express': 8,
        'market': 15,
        'school': 30,
        'kindergarten': 20,
    }
    
    # 等候时间（分钟）
    WAIT_TIME = {
        'convenience_store': 2,
        'pharmacy': 8,
        'supermarket': 10,
        'hospital': 20,
        'clinic': 10,
        'bank': 15,
        'atm': 2,
        'express': 5,
        'market': 8,
        'school': 0,
        'kindergarten': 0,
    }
    
    # 设施类型中英文映射
    FACILITY_NAMES = {
        'convenience_store': '便利店',
        'pharmacy': '药店',
        'supermarket': '超市',
        'hospital': '医院',
        'clinic': '诊所',
        'bank': '银行',
        'atm': 'ATM',
        'express': '快递',
        'market': '菜市场',
        'school': '学校',
        'kindergarten': '幼儿园',
    }
    
    # 社区类型中英文映射
    COMMUNITY_NAMES = {
        'urban_village': '城中村',
        'old_community': '老旧小区',
        'normal_community': '普通商品房',
        'high_end': '高端社区'
    }

config = StudyConfig()
print("配置加载完成")

### 2.2 加载OSM道路网络

In [ ]:
def load_road_network(data_dir):
    """加载OSM道路网络数据"""
    gpkg_file = data_dir / 'road_network.gpkg'
    
    if gpkg_file.exists():
        print(f"从GeoPackage加载道路网络: {gpkg_file}")
        gdf = gpd.read_file(gpkg_file)
        print(f"  加载完成: {len(gdf)} 条道路")
        return gdf
    else:
        print(f"警告: 道路网络文件不存在 - {gpkg_file}")
        return None

# 加载道路网络
road_gdf = load_road_network(DATA_DIR)
if road_gdf is not None:
    print(f"\n道路类型分布:")
    if 'highway' in road_gdf.columns:
        print(road_gdf['highway'].value_counts().head(10))
    print(f"\n坐标系: {road_gdf.crs}")

### 2.3 加载POI数据

In [ ]:
def load_poi_data(data_dir, source='osm'):
    """加载POI数据
    
    Args:
        data_dir: 数据目录
        source: 'osm' 或 'dianping' 或 'combined'
    """
    cache_dir = data_dir / 'osm_cache'
    
    if source == 'osm':
        file_path = cache_dir / 'poi_osm.csv'
    elif source == 'dianping':
        file_path = cache_dir / 'poi_dianping.csv'
    elif source == 'combined':
        file_path = cache_dir / 'facilities_combined.csv'
    else:
        raise ValueError(f"未知数据源: {source}")
    
    if file_path.exists():
        print(f"加载POI数据: {file_path}")
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        print(f"  加载完成: {len(df)} 条记录")
        return df
    else:
        print(f"警告: 文件不存在 - {file_path}")
        return None

# 加载POI数据
poi_df = load_poi_data(DATA_DIR, source='combined')

if poi_df is not None:
    print(f"\n数据列: {list(poi_df.columns)}")
    print(f"\n前5行:")
    display(poi_df.head())

### 2.4 加载小区AOI数据

In [ ]:
def load_community_data(data_dir):
    """加载小区AOI数据"""
    cache_dir = data_dir / 'osm_cache'
    file_path = cache_dir / 'communities.csv'
    
    if file_path.exists():
        print(f"加载小区数据: {file_path}")
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        print(f"  加载完成: {len(df)} 个小区")
        return df
    else:
        print(f"警告: 小区数据文件不存在 - {file_path}")
        return None

# 加载小区数据
community_df = load_community_data(DATA_DIR)

if community_df is not None:
    print(f"\n小区数据列: {list(community_df.columns)}")
    print(f"\n小区类型分布:")
    if 'community_type' in community_df.columns:
        print(community_df['community_type'].value_counts())

### 2.5 数据质量探索

In [ ]:
def explore_data_quality(df, name='数据集'):
    """数据质量探索报告"""
    print(f"\n{'='*60}")
    print(f"数据质量报告: {name}")
    print(f"{'='*60}")
    
    print(f"\n1. 基本信息:")
    print(f"   记录数: {len(df)}")
    print(f"   字段数: {len(df.columns)}")
    
    print(f"\n2. 缺失值统计:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        '缺失数': missing,
        '缺失率(%)': missing_pct
    })
    missing_df = missing_df[missing_df['缺失数'] > 0].sort_values('缺失率(%)', ascending=False)
    if len(missing_df) > 0:
        print(missing_df)
    else:
        print("   无缺失值")
    
    print(f"\n3. 数据类型:")
    print(df.dtypes)
    
    return missing_df

# 探索POI数据质量
if poi_df is not None:
    explore_data_quality(poi_df, 'POI设施')

---

<a id='3-POI清洗'></a>
## 3. POI数据清洗

### 3.1 坐标系统检查与转换

In [ ]:
def check_coordinate_system(df, lon_col='lng', lat_col='lat'):
    """检查坐标系统是否正确"""
    if lon_col not in df.columns or lat_col not in df.columns:
        print(f"警告: 找不到坐标列 {lon_col}, {lat_col}")
        return df
    
    # 检查坐标范围
    lon_min, lon_max = df[lon_col].min(), df[lon_col].max()
    lat_min, lat_max = df[lat_col].min(), df[lat_col].max()
    
    print(f"经度范围: {lon_min:.4f} ~ {lon_max:.4f}")
    print(f"纬度范围: {lat_min:.4f} ~ {lat_max:.4f}")
    
    # 判断坐标系
    if 113 < lon_min < 114 and 22 < lat_min < 23:
        print("✓ 坐标系正确: WGS84 (EPSG:4326)")
    elif 2500000 < lon_min < 2700000 and 500000 < lat_min < 600000:
        print("⚠ 检测到: GCJ-02 (高德坐标)")
    elif 11000000 < lon_min < 12500000 and 2500000 < lat_min < 4500000:
        print("⚠ 检测到: EPSG:3857 (Web Mercator)")
    else:
        print("⚠ 坐标系未知，可能需要转换")
    
    return df

def convert_coordinates(df, lon_col='lng', lat_col='lat', 
                        from_crs='EPSG:3857', to_crs='EPSG:4326'):
    """坐标系统转换
    
    Args:
        from_crs: 源坐标系
        to_crs: 目标坐标系
    """
    if from_crs == to_crs:
        print("坐标系相同，无需转换")
        return df
    
    print(f"坐标转换: {from_crs} -> {to_crs}")
    
    transformer = Transformer.from_crs(from_crs, to_crs, always_xy=True)
    
    # 转换坐标
    new_lon, new_lat = transformer.transform(
        df[lon_col].values, 
        df[lat_col].values
    )
    
    df[f'{lon_col}_original'] = df[lon_col]
    df[f'{lat_col}_original'] = df[lat_col]
    df[lon_col] = new_lon
    df[lat_col] = new_lat
    
    print(f"转换完成")
    return df

# 检查POI坐标
if poi_df is not None:
    poi_df = check_coordinate_system(poi_df)

### 3.2 南山区范围裁剪

In [ ]:
def clip_to_nanshan(df, bbox=None, lon_col='lng', lat_col='lat'):
    """裁剪到南山区范围
    
    Args:
        df: DataFrame
        bbox: 边界范围 {'north', 'south', 'east', 'west'}
        lon_col, lat_col: 坐标列名
    """
    if bbox is None:
        bbox = config.BBOX
    
    n_before = len(df)
    
    # 空间裁剪
    mask = (
        (df[lon_col] >= bbox['west']) & 
        (df[lon_col] <= bbox['east']) &
        (df[lat_col] >= bbox['south']) & 
        (df[lat_col] <= bbox['north'])
    )
    df_clipped = df[mask].copy()
    
    n_after = len(df_clipped)
    print(f"南山区裁剪: {n_before} -> {n_after} (剔除 {n_before - n_after})")
    
    return df_clipped

# 裁剪POI
if poi_df is not None:
    poi_df = clip_to_nanshan(poi_df, config.BBOX)
    print(f"\n裁剪后剩余: {len(poi_df)} 条")

### 3.3 异常值检测与处理

In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    """使用IQR方法检测异常值"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return lower, upper

def clean_poi_coordinates(df, lon_col='lng', lat_col='lat'):
    """清洗坐标异常值"""
    n_before = len(df)
    
    # 南山区大致范围
    valid_mask = (
        (df[lon_col] >= 113.80) & (df[lon_col] <= 114.10) &
        (df[lat_col] >= 22.40) & (df[lat_col] <= 22.60) &
        (df[lon_col].notna()) & (df[lat_col].notna())
    )
    
    df_clean = df[valid_mask].copy()
    
    n_after = len(df_clean)
    print(f"坐标清洗: {n_before} -> {n_after} (剔除 {n_before - n_after} 条异常坐标)")
    
    return df_clean

def clean_duplicates(df, lon_col='lng', lat_col='lat', 
                    name_col='name', keep='first'):
    """去重处理
    
    Args:
        df: DataFrame
        lon_col, lat_col, name_col: 经纬度列和名称列
        keep: 保留策略 'first' 或 'last'
    """
    n_before = len(df)
    
    # 四舍五入到5位小数，约11米精度
    df['_lng_round'] = df[lon_col].round(5)
    df['_lat_round'] = df[lat_col].round(5)
    
    # 按名称和位置去重
    df_clean = df.drop_duplicates(
        subset=['_lng_round', '_lat_round'],
        keep=keep
    )
    
    # 删除临时列
    df_clean = df_clean.drop(columns=['_lng_round', '_lat_round'])
    
    n_after = len(df_clean)
    print(f"去重处理: {n_before} -> {n_after} (去除 {n_before - n_after} 条重复记录)")
    
    return df_clean.reset_index(drop=True)

# 清洗POI数据
if poi_df is not None:
    poi_df = clean_poi_coordinates(poi_df)
    poi_df = clean_duplicates(poi_df)
    print(f"\n清洗后POI数量: {len(poi_df)}")

### 3.4 设施类型标准化

In [ ]:
# 设施类型映射表（根据原始数据中的分类映射到标准分类）
# 实际使用时需要根据你的原始数据中的分类进行调整

FACILITY_TYPE_MAPPING = {
    # 便利店类
    'convenience_store': ['便利店', '便利超市', '24小时便利店', '小型超市'],
    
    # 药店类
    'pharmacy': ['药店', '药房', '大药房', '医药店'],
    
    # 超市类
    'supermarket': ['超市', '大型超市', '百货超市', '生活超市'],
    
    # 医院类
    'hospital': ['医院', '三甲医院', '综合医院', '专科医院'],
    
    # 诊所类
    'clinic': ['诊所', '门诊', '卫生站', '社康中心', '社区卫生服务站'],
    
    # 银行类
    'bank': ['银行', '国有银行', '商业银行', '外资银行'],
    
    # ATM类
    'atm': ['ATM', '自动取款机', '自助银行'],
    
    # 快递类
    'express': ['快递', '菜鸟驿站', '快递驿站', '代收点', '快递柜'],
    
    # 菜市场类
    'market': ['菜市场', '农贸市场', '生鲜市场', '海鲜市场'],
    
    # 学校类
    'school': ['学校', '小学', '中学', '高中', '大学'],
    
    # 幼儿园类
    'kindergarten': ['幼儿园', '托儿所', '学前班']
}

def standardize_facility_type(df, type_col='facility_type', 
                               original_type_col='type'):
    """标准化设施类型
    
    Args:
        df: DataFrame
        type_col: 标准类型列名
        original_type_col: 原始类型列名
    """
    # 如果已有标准类型列，直接返回
    if type_col in df.columns:
        print(f"已有标准类型列: {type_col}")
        print(f"\n类型分布:")
        print(df[type_col].value_counts())
        return df
    
    # 创建反向映射表
    reverse_mapping = {}
    for std_type, keywords in FACILITY_TYPE_MAPPING.items():
        for keyword in keywords:
            reverse_mapping[keyword] = std_type
    
    # 映射类型
    def map_type(original_type):
        if pd.isna(original_type):
            return 'unknown'
        original_type = str(original_type).strip()
        for keyword, std_type in reverse_mapping.items():
            if keyword in original_type:
                return std_type
        return 'other'
    
    if original_type_col in df.columns:
        df[type_col] = df[original_type_col].apply(map_type)
    else:
        df[type_col] = 'unknown'
    
    print(f"\n标准化后的设施类型分布:")
    print(df[type_col].value_counts())
    
    return df

# 标准化设施类型
if poi_df is not None:
    poi_df = standardize_facility_type(poi_df)

---

<a id='4-小区AOI处理'></a>
## 4. 小区AOI数据处理

### 4.1 创建GeoDataFrame

In [ ]:
def df_to_geodataframe(df, lon_col='lng', lat_col='lat', crs='EPSG:4326'):
    """DataFrame转GeoDataFrame
    
    Args:
        df: DataFrame
        lon_col, lat_col: 经纬度列名
        crs: 坐标系
    """
    if df is None or len(df) == 0:
        print("警告: 空数据框")
        return None
    
    # 创建几何点
    geometry = [Point(lon, lat) for lon, lat in zip(df[lon_col], df[lat_col])]
    
    # 创建GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    
    print(f"GeoDataFrame创建完成: {len(gdf)} 个要素, CRS: {gdf.crs}")
    
    return gdf

# POI转GeoDataFrame
poi_gdf = df_to_geodataframe(poi_df)

# 小区转GeoDataFrame
community_gdf = df_to_geodataframe(community_df)

### 4.2 计算小区质心

In [ ]:
def calculate_centroids(gdf, lon_col='lng', lat_col='lat'):
    """计算质心坐标
    
    如果数据有geometry列（面要素），计算质心
    如果数据只有经纬度列，保持不变
    """
    if gdf is None:
        return None
    
    # 检查是否有面几何
    if 'geometry' in gdf.columns and len(gdf) > 0:
        geom_type = gdf.geometry.geom_type.iloc[0]
        
        if geom_type in ['Polygon', 'MultiPolygon']:
            print("检测到面要素，计算质心坐标...")
            centroids = gdf.geometry.centroid
            gdf[f'{lon_col}_centroid'] = centroids.x
            gdf[f'{lat_col}_centroid'] = centroids.y
            print("质心计算完成")
    else:
        print("无面几何，保持原有坐标")
    
    return gdf

# 计算小区质心
if community_gdf is not None:
    community_gdf = calculate_centroids(community_gdf)

### 4.3 社区类型识别

In [ ]:
def identify_community_type(df, type_col='community_type'):
    """识别/标准化社区类型
    
    规则:
    - 城中村: 面积较小(<5ha)，名称含"村"、"城"等
    - 老旧小区: 名称含"老"、"旧"、"宿舍"等
    - 高端社区: 名称含"豪宅"、"别墅"、"高端"等
    - 普通商品房: 其他
    """
    if type_col not in df.columns:
        df[type_col] = 'normal_community'  # 默认普通商品房
    
    def classify_community(row):
        name = str(row.get('name', ''))
        original_type = str(row.get(type_col, ''))
        
        # 如果已有分类
        if original_type in config.DETOUR_FACTORS:
            return original_type
        
        # 通过名称识别
        if any(kw in name for kw in ['村', '城中', '旧改']):
            return 'urban_village'
        elif any(kw in name for kw in ['老旧', '宿舍', '单位']):
            return 'old_community'
        elif any(kw in name for kw in ['豪宅', '别墅', '高端', '滨', '江景']):
            return 'high_end'
        else:
            return 'normal_community'
    
    df[type_col] = df.apply(classify_community, axis=1)
    
    print(f"\n社区类型分布:")
    type_counts = df[type_col].value_counts()
    
    # 显示中英文对照
    for ctype in type_counts.index:
        cn_name = config.COMMUNITY_NAMES.get(ctype, ctype)
        print(f"  {ctype} ({cn_name}): {type_counts[ctype]} 个")
    
    return df

# 识别社区类型
if community_gdf is not None:
    community_gdf = identify_community_type(community_gdf)

---

<a id='5-营业时间处理'></a>
## 5. 营业时间处理

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Tuple

@dataclass
class OpeningHours:
    """营业时间数据模型"""
    open_time: Optional[str] = None
    close_time: Optional[str] = None
    is_24h: bool = False
    night_service: bool = False
    business_period: str = ""
    service_hours: List[int] = field(default_factory=list)
    
    def to_hours(self) -> Tuple[Optional[int], Optional[int]]:
        """返回 (开放小时, 关闭小时)"""
        open_h, close_h = None, None
        if self.is_24h:
            return 0, 24
        if self.open_time:
            try:
                open_h = int(self.open_time.split(':')[0])
            except (ValueError, IndexError):
                pass
        if self.close_time:
            try:
                close_h = int(self.close_time.split(':')[0])
            except (ValueError, IndexError):
                pass
        return open_h, close_h
    
    def is_open_at(self, hour: int) -> bool:
        """检查特定小时是否营业"""
        if self.is_24h:
            return True
        if not self.service_hours:
            return True  # 无信息时假设营业
        return hour in self.service_hours
    
    @staticmethod
    def from_string(hours_str: str) -> 'OpeningHours':
        """从字符串解析营业时间"""
        if not hours_str or pd.isna(hours_str):
            return OpeningHours()
        
        hours_str = str(hours_str).strip()
        
        # 24小时识别
        if any(x in hours_str.lower() for x in ['24', '全天', '24h', '24小时']):
            return OpeningHours(
                is_24h=True,
                night_service=True,
                business_period=hours_str,
                service_hours=list(range(24))
            )
        
        result = OpeningHours(business_period=hours_str)
        
        # 解析时间范围
        try:
            for sep in ['-', '至', '~', '—']:
                if sep in hours_str:
                    parts = hours_str.split(sep)
                    if len(parts) == 2:
                        result.open_time = parts[0].strip()[:5]
                        result.close_time = parts[1].strip()[:5]
                        
                        open_h, close_h = result.to_hours()
                        if open_h is not None and close_h is not None:
                            if close_h > open_h:
                                result.service_hours = list(range(open_h, close_h))
                            else:  # 跨天
                                result.service_hours = list(range(open_h, 24)) + list(range(0, close_h))
                            
                            # 夜间服务识别
                            if close_h >= 20 or close_h < 6:
                                result.night_service = True
                        break
        except Exception as e:
            pass
        
        return result

def process_opening_hours(df, hours_col='opening_hours'):
    """处理营业时间数据"""
    print(f"处理营业时间数据...")
    
    if hours_col not in df.columns:
        print(f"  警告: 未找到营业时间列 '{hours_col}'")
        df['is_24h'] = False
        df['has_night_service'] = False
        return df
    
    # 解析营业时间
    hours_list = df[hours_col].apply(OpeningHours.from_string)
    
    # 提取关键属性
    df['is_24h'] = [h.is_24h for h in hours_list]
    df['has_night_service'] = [h.night_service for h in hours_list]
    df['open_hour'] = [h.to_hours()[0] for h in hours_list]
    df['close_hour'] = [h.to_hours()[1] for h in hours_list]
    
    print(f"\n营业时间统计:")
    print(f"  24小时营业: {df['is_24h'].sum()} ({df['is_24h'].mean()*100:.1f}%)")
    print(f"  夜间服务: {df['has_night_service'].sum()} ({df['has_night_service'].mean()*100:.1f}%)")
    
    return df

# 处理营业时间
if poi_df is not None:
    poi_df = process_opening_hours(poi_df)

### 5.1 按时段检查设施开放状态

In [ ]:
def check_period_availability(df, periods=None):
    """检查每个时段设施开放情况
    
    Args:
        df: DataFrame，需包含 is_24h, has_night_service, open_hour, close_hour
        periods: 时段定义
    """
    if periods is None:
        periods = config.PERIODS
    
    def is_available_in_period(row, period_start, period_end):
        # 24小时营业
        if row.get('is_24h', False):
            return True
        
        open_h = row.get('open_hour')
        close_h = row.get('close_hour')
        
        if open_h is None or close_h is None:
            return True  # 无信息时假设可用
        
        if close_h > open_h:  # 不过午夜
            return period_start >= open_h and period_end <= close_h
        else:  # 跨午夜
            return (period_start >= open_h) or (period_end <= close_h)
    
    for period_key, period_info in periods.items():
        period_start = period_info['start']
        period_end = period_info['end']
        
        col_name = f'available_{period_key}'
        df[col_name] = df.apply(
            lambda row: is_available_in_period(row, period_start, period_end),
            axis=1
        )
    
    print(f"\n各时段设施可用性:")
    for period_key, period_info in periods.items():
        col_name = f'available_{period_key}'
        available = df[col_name].sum()
        rate = df[col_name].mean() * 100
        print(f"  {period_info['name']} ({period_info['description']}): "
              f"{available}/{len(df)} ({rate:.1f}%)")
    
    return df

# 检查各时段可用性
if poi_df is not None:
    poi_df = check_period_availability(poi_df)

---

<a id='6-空间分析'></a>
## 6. 空间分析

### 6.1 设施密度分析

In [ ]:
def analyze_facility_density(gdf, facility_type_col='facility_type'):
    """分析各类设施密度分布"""
    print(f"\n{'='*60}")
    print(f"设施密度分析")
    print(f"{'='*60}")
    
    if gdf is None:
        print("无数据")
        return None
    
    # 按类型统计
    if facility_type_col not in gdf.columns:
        print("警告: 未找到设施类型列")
        return gdf[facility_type_col].value_counts() if facility_type_col in gdf.columns else None
    
    type_counts = gdf[facility_type_col].value_counts()
    
    print(f"\n设施类型统计:")
    for ftype in type_counts.index:
        fname = config.FACILITY_NAMES.get(ftype, ftype)
        count = type_counts[ftype]
        print(f"  {fname}: {count} 个")
    
    # 计算总设施密度
    bbox = gdf.total_bounds  # [minx, miny, maxx, maxy]
    area_km2 = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]) * 111 * 111 / 2  # 粗略估算
    density = len(gdf) / max(area_km2, 1)
    
    print(f"\n总体设施密度: {density:.1f} 个/km²")
    
    return type_counts

# 设施密度分析
if poi_gdf is not None:
    analyze_facility_density(poi_gdf)

### 6.2 空间连接：小区与最近设施

In [ ]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """计算两点间Haversine距离(km)"""
    R = 6371  # 地球半径
    
    lat1, lat2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    return R * c

def find_nearest_facilities(community_gdf, facility_gdf, 
                           facility_type=None, n=1):
    """查找每个小区最近的设施
    
    Args:
        community_gdf: 小区GeoDataFrame
        facility_gdf: 设施GeoDataFrame
        facility_type: 设施类型过滤
        n: 返回最近n个设施
    """
    # 过滤设施类型
    if facility_type:
        facilities = facility_gdf[facility_gdf['facility_type'] == facility_type].copy()
    else:
        facilities = facility_gdf.copy()
    
    if len(facilities) == 0:
        print(f"警告: 无{facility_type}类型设施")
        return None
    
    print(f"查找最近的 {n} 个 {facility_type}... ({len(facilities)} 个设施)")
    
    # 构建KD树
    facility_coords = np.array(list(zip(facilities.geometry.x, facilities.geometry.y)))
    tree = cKDTree(facility_coords)
    
    # 获取小区坐标
    if 'geometry' in community_gdf.columns:
        community_coords = np.array(list(zip(
            community_gdf.geometry.x,
            community_gdf.geometry.y
        )))
        id_col = 'cid' if 'cid' in community_gdf.columns else community_gdf.index.name
        if id_col is None:
            community_gdf = community_gdf.reset_index(drop=True)
            community_gdf['cid'] = range(len(community_gdf))
            id_col = 'cid'
    else:
        community_coords = np.array(list(zip(
            community_gdf['lng'],
            community_gdf['lat']
        )))
        id_col = 'cid' if 'cid' in community_gdf.columns else community_gdf.index.name
    
    # 查询最近设施
    max_k = min(n, len(facilities))
    distances, indices = tree.query(community_coords, k=max_k)
    
    # 提取结果
    results = []
    for i, (comm_idx, dist) in enumerate(zip(range(len(community_gdf)), distances)):
        if n == 1:
            dist = [dist]
            indices_i = [indices]
        else:
            indices_i = indices[i] if len(indices.shape) > 1 else indices
        
        nearest_fid = None
        nearest_dist = None
        
        for d, idx in zip(dist, indices_i):
            if idx < len(facilities):
                nearest_fid = facilities.iloc[idx]['fid'] if 'fid' in facilities.columns else idx
                nearest_dist = d
                break
        
        results.append({
            'cid': community_gdf.iloc[comm_idx]['cid'] if 'cid' in community_gdf.columns else comm_idx,
            f'nearest_{facility_type}_fid': nearest_fid,
            f'nearest_{facility_type}_dist_km': nearest_dist
        })
    
    return pd.DataFrame(results)

# 查找各类设施最近距离
if community_gdf is not None and poi_gdf is not None:
    facility_types = list(config.FACILITY_NAMES.keys())
    
    # 确保community_gdf有cid列
    if 'cid' not in community_gdf.columns:
        community_gdf['cid'] = range(len(community_gdf))
    
    # 确保poi_gdf有fid列
    if 'fid' not in poi_gdf.columns:
        poi_gdf['fid'] = range(len(poi_gdf))
    
    nearest_results = {}
    for ftype in facility_types:
        fname = config.FACILITY_NAMES.get(ftype, ftype)
        result = find_nearest_facilities(community_gdf, poi_gdf, ftype)
        if result is not None:
            nearest_results[ftype] = result
    
    # 合并结果
    if nearest_results:
        community_access = community_gdf.copy()
        for ftype, result_df in nearest_results.items():
            community_access = community_access.merge(
                result_df, on='cid', how='left'
            )
        print(f"\n合并后字段: {list(community_access.columns)[:20]}...")

---

<a id='7-可达性计算'></a>
## 7. 时间可达性计算

In [ ]:
def calculate_travel_time(distance_km, walk_speed_kmh=5.0):
    """计算步行时间(分钟)
    
    Args:
        distance_km: 距离(km)
        walk_speed_kmh: 步行速度(km/h)
    """
    speed_m_per_min = walk_speed_kmh * 1000 / 60
    return distance_km * 1000 / speed_m_per_min

def calculate_accessibility_metrics(community_df, facility_types, 
                                    config=StudyConfig):
    """计算时间可达性指标
    
    公式:
    - MTI (Movement Time Index) = 步行时间 × 曲折系数 / 15分钟阈值
    - WTI (Wait Time Index) = 等候时间 / 15分钟阈值
    - TMI (Service Time Index) = 服务时间 / 15分钟阈值
    - CTPI (Comprehensive Time Poverty Index) = (MTI + WTI + TMI) / 3
    """
    print(f"\n{'='*60}")
    print(f"时间可达性计算")
    print(f"{'='*60}")
    
    df = community_df.copy()
    periods = list(config.PERIODS.keys())
    
    for ftype in facility_types:
        dist_col = f'nearest_{ftype}_dist_km'
        
        if dist_col not in df.columns:
            continue
        
        # 步行时间
        df[f'{ftype}_travel_time'] = df[dist_col].apply(
            lambda d: calculate_travel_time(d, config.WALK_SPEED_KMH) if pd.notna(d) else np.nan
        )
        
        # 曲折系数
        detour = df['community_type'].map(config.DETOUR_FACTORS).fillna(1.1)
        
        # 等候时间
        wait_time = config.WAIT_TIME.get(ftype, 5)
        
        # 服务时间
        service_time = config.SERVICE_DURATION.get(ftype, 15)
        
        # 各时段可达性
        for period in periods:
            period_info = config.PERIODS[period]
            
            # 获取时段可用性列
            avail_col = f'available_{period}'
            
            # 计算综合时间（如果设施在该时段不可用，加上惩罚时间）
            df[f'{ftype}_{period}_time'] = np.where(
                df[dist_col].notna(),
                df[f'{ftype}_travel_time'] * detour + wait_time + service_time,
                np.nan
            )
            
            # 计算CTPI
            mti = df[f'{ftype}_{period}_time'] / config.ACCESSIBILITY_THRESHOLD
            wti = wait_time / config.ACCESSIBILITY_THRESHOLD
            tmi = service_time / config.ACCESSIBILITY_THRESHOLD
            df[f'{ftype}_{period}_ctpi'] = (mti + wti + tmi) / 3
    
    print(f"计算完成")
    
    return df

# 计算可达性
if 'community_access' in dir():
    accessibility_df = calculate_accessibility_metrics(
        community_access, 
        list(config.FACILITY_NAMES.keys())
    )
    print(f"\n可达性结果维度: {accessibility_df.shape}")

---

<a id='8-贫困指数分析'></a>
## 8. 时空贫困指数分析

In [ ]:
def calculate_period_averages(df, facility_types, periods):
    """计算各时段平均CTPI"""
    
    for period in periods:
        ctpi_cols = [f'{ftype}_{period}_ctpi' for ftype in facility_types 
                    if f'{ftype}_{period}_ctpi' in df.columns]
        
        if ctpi_cols:
            df[f'{period}_avg_ctpi'] = df[ctpi_cols].mean(axis=1)
            
            # 可达设施数量（CTPI <= 1.0）
            df[f'{period}_accessible_count'] = (df[ctpi_cols] <= 1.0).sum(axis=1)
    
    return df

def analyze_time_poverty(df, periods):
    """时空贫困分析"""
    print(f"\n{'='*60}")
    print(f"时空贫困分析报告")
    print(f"{'='*60}")
    
    for period in periods:
        avg_col = f'{period}_avg_ctpi'
        if avg_col not in df.columns:
            continue
        
        period_name = config.PERIODS[period]['name']
        
        print(f"\n【{period_name}时段】")
        print(f"  平均CTPI: {df[avg_col].mean():.3f}")
        print(f"  中位数CTPI: {df[avg_col].median():.3f}")
        print(f"  标准差: {df[avg_col].std():.3f}")
        print(f"  最小值: {df[avg_col].min():.3f}")
        print(f"  最大值: {df[avg_col].max():.3f}")
        
        # 可达率
        accessible_rate = (df[avg_col] <= 1.0).mean() * 100
        print(f"  可达率(CTPI≤1): {accessible_rate:.1f}%")
    
    # 白天vs夜间对比
    print(f"\n【时段对比】")
    if 'day_avg_ctpi' in df.columns and 'evening_avg_ctpi' in df.columns:
        diff = (df['evening_avg_ctpi'] - df['day_avg_ctpi']).mean()
        print(f"  夜间相对白天CTPI变化: {diff:+.3f}")
    
    if 'day_avg_ctpi' in df.columns and 'night_avg_ctpi' in df.columns:
        diff = (df['night_avg_ctpi'] - df['day_avg_ctpi']).mean()
        print(f"  深夜相对白天CTPI变化: {diff:+.3f}")
    
    return df

# 计算时段平均
if 'accessibility_df' in dir():
    accessibility_df = calculate_period_averages(
        accessibility_df,
        list(config.FACILITY_NAMES.keys()),
        list(config.PERIODS.keys())
    )
    accessibility_df = analyze_time_poverty(
        accessibility_df,
        list(config.PERIODS.keys())
    )

### 8.1 社区类型贫困对比

In [ ]:
def compare_community_types(df, periods):
    """社区类型贫困对比分析"""
    print(f"\n{'='*60}")
    print(f"社区类型时空贫困对比")
    print(f"{'='*60}")
    
    results = []
    
    for ctype in config.DETOUR_FACTORS.keys():
        cn_name = config.COMMUNITY_NAMES.get(ctype, ctype)
        subset = df[df['community_type'] == ctype]
        
        if len(subset) == 0:
            continue
        
        row = {'community_type': ctype, 'cn_name': cn_name, 'count': len(subset)}
        
        for period in periods:
            avg_col = f'{period}_avg_ctpi'
            if avg_col in subset.columns:
                row[f'{period}_mean'] = subset[avg_col].mean()
                row[f'{period}_std'] = subset[avg_col].std()
        
        results.append(row)
    
    result_df = pd.DataFrame(results)
    
    print(f"\n各社区类型平均CTPI:")
    display(result_df)
    
    return result_df

# 社区类型对比
if 'accessibility_df' in dir():
    community_comparison = compare_community_types(
        accessibility_df, 
        list(config.PERIODS.keys())
    )

---

<a id='9-可视化输出'></a>
## 9. 可视化与输出

### 9.1 设施分布可视化

In [ ]:
def plot_facility_distribution(gdf, facility_types, config=StudyConfig):
    """设施分布可视化"""
    
    n_types = len(facility_types)
    n_cols = 3
    n_rows = math.ceil(n_types / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    if n_types > 1:
        axes = axes.flatten()
    else:
        axes = [axes]
    
    colors = plt.cm.Set2(np.linspace(0, 1, max(n_types, 1)))
    
    for idx, ftype in enumerate(facility_types):
        ax = axes[idx]
        fname = config.FACILITY_NAMES.get(ftype, ftype)
        
        if ftype in gdf['facility_type'].values:
            subset = gdf[gdf['facility_type'] == ftype]
            ax.scatter(subset.geometry.x, subset.geometry.y, 
                      c=[colors[idx]], s=20, alpha=0.6, label=fname)
        
        ax.set_title(f'{fname}分布 ({len(gdf[gdf["facility_type"]==ftype]) if "facility_type" in gdf.columns else 0}个)', fontsize=11)
        ax.set_xlabel('经度')
        ax.set_ylabel('纬度')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # 隐藏空白子图
    for idx in range(len(facility_types), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'facility_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n图表已保存: {OUTPUT_DIR / 'facility_distribution.png'}")

# 绘制设施分布
if poi_gdf is not None:
    plot_facility_distribution(poi_gdf, list(config.FACILITY_NAMES.keys()))

### 9.2 时空可达性地图

In [ ]:
def plot_accessibility_map(df, periods, output_dir=OUTPUT_DIR):
    """时空可达性空间分布图"""
    
    n_periods = len(periods)
    fig, axes = plt.subplots(1, n_periods, figsize=(6*n_periods, 5))
    if n_periods == 1:
        axes = [axes]
    
    # 设置colormap
    cmap = LinearSegmentedColormap.from_list('ctpi', 
                                            ['#2ecc71', '#f1c40f', '#e74c3c'])
    vmin, vmax = 0, 2.0
    
    for idx, period in enumerate(periods):
        ax = axes[idx]
        col = f'{period}_avg_ctpi'
        
        if col not in df.columns:
            continue
        
        period_name = config.PERIODS[period]['name']
        
        # 绘制散点图
        scatter = ax.scatter(
            df['lng'], df['lat'], 
            c=df[col], 
            cmap=cmap, 
            s=80, 
            alpha=0.7,
            vmin=vmin, vmax=vmax,
            edgecolors='black', linewidths=0.3
        )
        
        ax.set_title(f'{period_name}时段\nCTPI空间分布', fontsize=12, fontweight='bold')
        ax.set_xlabel('经度')
        ax.set_ylabel('纬度')
        ax.grid(True, alpha=0.3)
        
        # 添加colorbar
        cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
        cbar.set_label('CTPI')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'accessibility_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"图表已保存: {output_dir / 'accessibility_map.png'}")

# 绘制可达性地图
if 'accessibility_df' in dir():
    plot_accessibility_map(accessibility_df, list(config.PERIODS.keys()))

### 9.3 白天vs夜间对比

In [ ]:
def plot_day_night_comparison(df, output_dir=OUTPUT_DIR):
    """白天vs夜间对比图"""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 左图：白天CTPI
    if 'day_avg_ctpi' in df.columns:
        scatter1 = axes[0].scatter(
            df['lng'], df['lat'], c=df['day_avg_ctpi'],
            cmap='RdYlGn_r', s=80, alpha=0.7, vmin=0, vmax=2
        )
        axes[0].set_title('白天时段 (08:00-18:00)\nCTPI分布', fontweight='bold')
        plt.colorbar(scatter1, ax=axes[0], shrink=0.6, label='CTPI')
    
    # 中图：夜间CTPI
    if 'evening_avg_ctpi' in df.columns:
        scatter2 = axes[1].scatter(
            df['lng'], df['lat'], c=df['evening_avg_ctpi'],
            cmap='RdYlGn_r', s=80, alpha=0.7, vmin=0, vmax=2
        )
        axes[1].set_title('夜间时段 (18:00-22:00)\nCTPI分布', fontweight='bold')
        plt.colorbar(scatter2, ax=axes[1], shrink=0.6, label='CTPI')
    
    # 右图：CTPI变化量
    if 'day_avg_ctpi' in df.columns and 'evening_avg_ctpi' in df.columns:
        diff = df['evening_avg_ctpi'] - df['day_avg_ctpi']
        scatter3 = axes[2].scatter(
            df['lng'], df['lat'], c=diff,
            cmap='RdBu', s=80, alpha=0.7, vmin=-0.5, vmax=0.5
        )
        axes[2].set_title('CTPI变化量\n(夜间-白天)', fontweight='bold')
        plt.colorbar(scatter3, ax=axes[2], shrink=0.6, label='CTPI变化')
    
    for ax in axes:
        ax.set_xlabel('经度')
        ax.set_ylabel('纬度')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'day_night_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"图表已保存: {output_dir / 'day_night_comparison.png'}")

# 绘制对比图
if 'accessibility_df' in dir():
    plot_day_night_comparison(accessibility_df)

### 9.4 社区类型对比

In [ ]:
def plot_community_comparison(comparison_df, output_dir=OUTPUT_DIR):
    """社区类型对比柱状图"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    types = comparison_df['cn_name'].tolist()
    x = np.arange(len(types))
    width = 0.25
    
    colors = {'day': '#3498db', 'evening': '#e74c3c', 'night': '#9b59b6'}
    
    for i, period in enumerate(['day', 'evening', 'night']):
        col = f'{period}_mean'
        if col in comparison_df.columns:
            period_name = config.PERIODS[period]['name']
            ax.bar(x + (i-1)*width, comparison_df[col], width, 
                   label=f'{period_name}时段', color=colors[period], alpha=0.8)
    
    ax.set_ylabel('平均CTPI', fontsize=12)
    ax.set_title('不同社区类型时空贫困指数对比', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(types, fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='可达性阈值')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'community_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"图表已保存: {output_dir / 'community_comparison.png'}")

# 绘制社区对比图
if 'community_comparison' in dir():
    plot_community_comparison(community_comparison)

### 9.5 输出数据文件

In [ ]:
def export_results(df, poi_gdf, community_gdf, output_dir=OUTPUT_DIR):
    """导出分析结果"""
    print(f"\n{'='*60}")
    print(f"导出分析结果")
    print(f"{'='*60}")
    
    # 1. 设施数据
    if poi_df is not None:
        poi_output = output_dir / 'facilities_nanshan.csv'
        poi_df.to_csv(poi_output, index=False, encoding='utf-8-sig')
        print(f"  设施数据: {poi_output}")
    
    # 2. 小区可达性结果
    if 'accessibility_df' in dir():
        acc_output = output_dir / 'accessibility_results.csv'
        # 转换为非GeoDataFrame格式保存
        acc_export = accessibility_df.drop(columns=['geometry'], errors='ignore')
        acc_export.to_csv(acc_output, index=False, encoding='utf-8-sig')
        print(f"  可达性结果: {acc_output}")
    
    # 3. 统计汇总
    if 'community_comparison' in dir():
        stats_output = output_dir / 'community_statistics.csv'
        community_comparison.to_csv(stats_output, index=False, encoding='utf-8-sig')
        print(f"  社区统计: {stats_output}")
    
    # 4. GeoJSON导出（用于GIS软件）
    if poi_gdf is not None:
        poi_geojson = output_dir / 'facilities_nanshan.geojson'
        poi_gdf.to_file(poi_geojson, driver='GeoJSON')
        print(f"  设施GeoJSON: {poi_geojson}")
    
    if community_gdf is not None:
        comm_geojson = output_dir / 'communities_nanshan.geojson'
        community_gdf.to_file(comm_geojson, driver='GeoJSON')
        print(f"  小区GeoJSON: {comm_geojson}")
    
    print(f"\n所有结果已保存到: {output_dir}")

# 导出结果
export_results(accessibility_df if 'accessibility_df' in dir() else None, 
              poi_gdf, community_gdf)

---

## 10. ArcGIS处理指南

### 10.1 数据导入ArcGIS Pro

**步骤1：导入GeoJSON文件**

```
1. 设施点数据 (facilities_nanshan.geojson)
   - 菜单: 文件 -> 导入地图 -> 选择GeoJSON文件
   - 或使用: 工具箱 -> 数据管理 -> 要素类 -> JSON转要素类

2. 小区面数据 (communities_nanshan.geojson)
   同上

3. 设置坐标系
   右键图层 -> 属性 -> 坐标系 -> 选择 WGS 1984 (EPSG:4326)
```

### 10.2 核心分析工具

**工具1：近邻分析**
```
位置: 分析工具 -> 邻域分析 -> 生成连接表
输入: 小区点要素
连接要素: 设施点要素
输出: 包含距离和方向的新图层
```

**工具2：缓冲区分析**
```
位置: 分析工具 -> 邻域分析 -> 缓冲区
输入要素: 设施点
距离: 500米 (步行15分钟约750米，取保守值500米)
融合类型: ALL (合并所有缓冲区)
```

**工具3：核密度分析**
```
位置: Spatial Analyst -> 密度分析 -> 核密度
输入: 设施点
Population字段: None (每个点权重相同)
搜索半径: 500米
输出像元大小: 50米
```

**工具4：空间连接**
```
位置: 分析工具 -> 叠加分析 -> 空间连接
目标要素: 小区面
连接要素: 设施点
连接操作: ONE_TO_ONE
匹配选项: INTERSECT
```

### 10.3 符号化设置

**设施分布图**
```
- 使用唯一值符号化
- 字段: facility_type
- 每种类型设置不同颜色
```

**CTPI分布图**
```
- 使用渐变颜色符号化
- 字段: day_avg_ctpi / evening_avg_ctpi / night_avg_ctpi
- 分类: 分段设色
  - 0-0.5: 绿色 (低贫困)
  - 0.5-1.0: 黄色 (中贫困)
  - 1.0-1.5: 橙色 (高贫困)
  - >1.5: 红色 (极端贫困)
```

### 10.4 地图布局建议

```
1. 添加图例
   插入 -> 图例
   选择需要显示的图层

2. 添加比例尺和指北针
   插入 -> 比例尺 / 指北针

3. 添加标题
   插入 -> 标题
   建议标题: 南山区公共服务时空可达性分析图

4. 添加数据来源说明
   插入 -> 文本
   数据来源: OSM, 高德地图, 大众点评等
```

---

## 分析完成！

### 输出文件清单

| 文件名 | 描述 |
|-------|------|
| `facilities_nanshan.csv` | 南山区设施数据 |
| `facilities_nanshan.geojson` | 设施GeoJSON (ArcGIS可导入) |
| `communities_nanshan.geojson` | 小区GeoJSON |
| `accessibility_results.csv` | 可达性分析结果 |
| `community_statistics.csv` | 社区类型统计 |
| `facility_distribution.png` | 设施分布图 |
| `accessibility_map.png` | 时空可达性地图 |
| `day_night_comparison.png` | 白天夜间对比图 |
| `community_comparison.png` | 社区类型对比图 |

### 下一步

1. 在ArcGIS Pro中导入GeoJSON文件
2. 进行更精细的空间分析
3. 制作高质量专题地图
4. 撰写研究报告